# CSMP 150 — PCA and tICA homework

Compare fluctuations and slow relaxation in a one-dimensional four-well potential and a two-dimensional eight-well potential.

The cells below supply the potentials, reversible Metropolis MC simulations, trajectories, and spatial-bin features. Time is measured in **MC steps**, not physical time. The 2D potential is a teaching extension of the lecture's scaled Prinz-style potential. The models use different grids, so their numerical MC times are not directly comparable.

Run the supplied cells first, then complete the analysis below for **both systems**. Raw position alone provides only one feature in 1D and two in 2D. Use the supplied spatial-bin indicator features to obtain four nonconstant components; interpret the resulting PCA and tICA as analyses of these features.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("numba") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.fft import rfft, irfft, next_fast_len
from numba import njit
from IPython.display import display
from pathlib import Path
import pandas as pd
OUTPUT = Path("tica_figures")
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 120})
COLORS = ["#d77b0b", "#71952a", "#007a9b", "#9452a3"]


## 1. Define and show the potentials

$$V_x(x)=2x^8+0.8e^{-80x^2}+0.2e^{-80(x-0.5)^2}+0.5e^{-40(x+0.5)^2},$$
$$V_y(y)=0.6\left[\left(\frac{y}{0.55}\right)^2-1\right]^2,\qquad V(x,y)=V_x(x)+V_y(y).$$

We restrict coordinates to $[-1,1]$ and use $\beta=5$. On each uniform grid, the equilibrium probability is $\pi_j\propto e^{-\beta V_j}$. The 2D equilibrium free energy in thermal units is $\beta F=\beta V+\mathrm{constant}$.

This is the scaled Prinz-style potential used in the lecture; the separable double-well extension is a teaching model.

In [ ]:
BETA = 5.0
def prinz(x):
    return (2*x**8 + .8*np.exp(-80*x*x)
            + .2*np.exp(-80*(x-.5)**2) + .5*np.exp(-40*(x+.5)**2))
def double_well(y):
    return .6*((y/.55)**2-1)**2

grid1 = np.linspace(-1, 1, 160)
grid2 = np.linspace(-1, 1, 40)
xx, yy = np.meshgrid(grid2, grid2)
energy1 = prinz(grid1)
energy2 = prinz(xx) + double_well(yy)
fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax[0].plot(grid1, energy1, color="black", lw=2)
ax[0].set(xlabel="x", ylabel="Potential energy", title="1D: four wells")
cf = ax[1].contourf(xx, yy, BETA*(energy2-energy2.min()), levels=np.linspace(0,12,25), extend="max", cmap="viridis")
ax[1].contour(xx, yy, BETA*(energy2-energy2.min()), levels=[1,2,3,4,6,8,10], colors="white", linewidths=.4, alpha=.6)
ax[1].set(xlabel="x", ylabel="y", title="2D: eight wells", aspect="equal")
fig.colorbar(cf, ax=ax[1], label="Free energy / kBT (minimum = 0)")
fig.savefig(OUTPUT/"potentials.png", dpi=220)
plt.show()


## 2. Generate reversible MC trajectories

A symmetric nearest-neighbor proposal is accepted with
$$A(i\to j)=\min\{1,e^{-\beta(V_j-V_i)}\}.$$

In 1D, propose left/right with probability 1/4 each and stay with probability 1/2. In 2D, first choose x or y equally, then use the same proposal. Rejections retain the current position and count toward elapsed time. This explicit stay probability makes the chains lazy.

We start from an exact equilibrium draw so no burn-in is needed. A single trajectory still fluctuates and must be long enough to sample exchange among wells. Store every 5th step; the fitting lag is 1,000 MC steps, or 200 stored frames. Adjusting the proposal or grid changes the kinetics.

In [ ]:
@njit
def simulate(energy, n_steps, stride, seed, initial, two_dimensional):
    np.random.seed(seed)
    ny, nx = energy.shape
    state = initial
    traj = np.empty(n_steps//stride+1, dtype=np.int32)
    traj[0] = state
    accepted = 0
    for t in range(1, n_steps+1):
        iy, ix = state//nx, state%nx
        # Half of the proposals are explicit self transitions.
        r = np.random.random()
        if r < .5:
            step = -1 if r < .25 else 1
            jy, jx = iy, ix
            if two_dimensional and np.random.random() < .5:
                jy += step
            else:
                jx += step
            if 0 <= jx < nx and 0 <= jy < ny:
                if np.random.random() < np.exp(min(0., -BETA*(energy[jy,jx]-energy[iy,ix]))):
                    state = jy*nx+jx
                    accepted += 1
        if t % stride == 0:
            traj[t//stride] = state
    return traj, accepted/n_steps

STRIDE = 5
LAG_MC = 1000
LAG = LAG_MC//STRIDE
systems = {}
for name, energy, n_steps, seed, two_d in [
    ("1D", energy1[None,:], 8_000_000, 150, False),
    ("2D", energy2, 4_000_000, 151, True)]:
    pi = np.exp(-BETA*(energy.ravel()-energy.min()))
    pi /= pi.sum()
    initial = np.random.default_rng(seed).choice(len(pi), p=pi)
    traj, acceptance = simulate(energy, n_steps, STRIDE, seed, initial, two_d)
    systems[name] = dict(energy=energy, pi=pi, trajectory=traj, two_d=two_d)
    print(f"{name}: {len(traj):,} stored frames; {n_steps:,} MC steps; actual move fraction {acceptance:.3f}")


## 3. Inspect trajectory sampling and define the features

Use 32 spatial bins in 1D and a 10 × 10 grid in 2D. A frame has feature value 1 in its occupied bin and 0 elsewhere. We compute covariances directly from counts, without allocating a large dense one-hot trajectory.

Check both the spatial occupancy and the coordinate time trace. A matching equilibrium histogram alone does not establish accurate slow kinetics.

In [ ]:
for name, d in systems.items():
    ny, nx = d["energy"].shape
    if name == "1D":
        mapping = np.arange(nx)//5  # 160 sites -> 32 feature bins
    else:
        iy, ix = np.indices((ny,nx))
        mapping = ((iy//4)*10 + ix//4).ravel()  # 40x40 -> 10x10
    d["mapping"] = mapping
    d["features"] = mapping[d["trajectory"]]
    d["n_features"] = int(mapping.max()+1)

fig, ax = plt.subplots(2,2, figsize=(11,7), constrained_layout=True)
d=systems["1D"]
obs=np.bincount(d["trajectory"], minlength=160)/len(d["trajectory"])
ax[0,0].plot(grid1, d["pi"], "k--", label="Equilibrium")
ax[0,0].plot(grid1, obs, color="#007a9b", label="Sampled")
ax[0,0].set(xlabel="x", ylabel="Probability per grid site", title="1D sampling")
ax[0,0].legend()
show=20_000
ax[0,1].plot(np.arange(show)*STRIDE, grid1[d["trajectory"][:show]], lw=.6)
ax[0,1].set(xlabel="MC steps", ylabel="x", title="First 100,000 MC steps")
d=systems["2D"]
obs=np.bincount(d["trajectory"], minlength=1600).reshape(40,40)
cf=ax[1,0].pcolormesh(xx, yy, obs/obs.sum(), shading="nearest", cmap="viridis")
ax[1,0].set(xlabel="x", ylabel="y", title="2D sampled occupancy", aspect="equal")
fig.colorbar(cf, ax=ax[1,0], label="Probability per grid site")
ax[1,1].plot(np.arange(show)*STRIDE, grid2[d["trajectory"][:show]%40], label="x", lw=.6)
ax[1,1].plot(np.arange(show)*STRIDE, grid2[d["trajectory"][:show]//40], label="y", lw=.6)
ax[1,1].set(xlabel="MC steps", ylabel="Coordinate", title="First 100,000 MC steps")
ax[1,1].legend()
fig.savefig(OUTPUT/"sampling.png", dpi=200)
plt.show()


## 1. Inspect the sampling

Plot the potentials and sampled populations. Compare the sampled distribution with equilibrium. Describe which wells are visited and how you would assess adequate sampling.

## 2. Construct covariance matrices

Using the supplied feature labels, compute centered instantaneous and lagged covariance matrices at a lag of 1,000 MC steps. Convert MC steps to saved-frame offsets. Use a consistent reversible, paired estimator for both matrices. State your centering and normalization conventions; report matrix dimensions.

## 3. Compute PCA and tICA

Solve the PCA eigenproblem and tICA generalized eigenproblem. Remove the constant, zero-variance direction. Retain the four largest PCA eigenvalues and four largest nonconstant tICA eigenvalues. Normalize the PCA vectors to Euclidean unit length and the tICA vectors to unit covariance norm. Verify the eigenvalue equations numerically.

## 4. Visualize the four components

Plot all four PCs and all four tICs as functions of position: curves or steps in 1D, and spatial color maps in 2D. Report PCA variance fractions and tICA eigenvalues. Explain which population contrasts each component represents and identify poorly resolved modes.

## 5. Project the trajectories

Center the features and project every saved frame onto each of the four PCs and four tICs. Plot representative time traces on a shared MC-step axis. Standardize the displayed PC traces when comparing their temporal behavior with the tICs; explain the distinction between an eigenvector and its trajectory projection.

## 6. Compute autocorrelations and correlation times

Calculate normalized autocorrelation functions for all eight projected coordinates in each system. Estimate an integrated autocorrelation time for every PC and tIC, stating the integration convention and truncation rule. For positive tICA eigenvalues below one, also compute the implied relaxation time −τ / ln(λ). Compare the two kinds of time estimates and discuss when they need not agree.